In [1]:
import pandas as pd
student_data = pd.read_csv('student_depression_dataset.csv')

print(student_data.shape)
print(student_data.dtypes)
student_data.head(10)

(27901, 18)
id                                         int64
Gender                                    object
Age                                      float64
City                                      object
Profession                                object
Academic Pressure                        float64
Work Pressure                            float64
CGPA                                     float64
Study Satisfaction                       float64
Job Satisfaction                         float64
Sleep Duration                            object
Dietary Habits                            object
Degree                                    object
Have you ever had suicidal thoughts ?     object
Work/Study Hours                         float64
Financial Stress                          object
Family History of Mental Illness          object
Depression                                 int64
dtype: object


,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,'5-6 hours',Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,'5-6 hours',Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,'Less than 5 hours',Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,'7-8 hours',Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,'5-6 hours',Moderate,M.Tech,Yes,1.0,1.0,No,0
5,33,Male,29.0,Pune,Student,2.0,0.0,5.70,3.0,0.0,'Less than 5 hours',Healthy,PhD,No,4.0,1.0,No,0
6,52,Male,30.0,Thane,Student,3.0,0.0,9.54,4.0,0.0,'7-8 hours',Healthy,BSc,No,1.0,2.0,No,0
7,56,Female,30.0,Chennai,Student,2.0,0.0,8.04,4.0,0.0,'Less than 5 hours',Unhealthy,'Class 12',No,0.0,1.0,Yes,0
8,59,Male,28.0,Nagpur,Student,3.0,0.0,9.79,1.0,0.0,'7-8 hours',Moderate,B.Ed,Yes,12.0,3.0,No,1
9,62,Male,31.0,Nashik,Student,2.0,0.0,8.38,3.0,0.0,'Less than 5 hours',Moderate,LLB,Yes,2.0,5.0,No,1


Since project is about students only we remove the non student professions

In [2]:
student_data = student_data[student_data['Profession'] == 'Student']

Remove rows with City column data that aren't cities

In [3]:
invalid_city_values = ['Saanvi', 'Bhavna', 'City', 'Harsha', 'Kibara', 'Nandini',
                       'Nalini', 'Mihir', 'Nalyan', 'M.Com', 'ME', 'Rashi', 'Gaurav',
                       'Reyansh', 'Harsh', 'Vaanya', 'Mira', "'Less than 5 Kalyan'",
                       '3.0', "'Less Delhi'", 'M.Tech', 'Khaziabad']

student_data = student_data[~student_data['City'].isin(invalid_city_values)]

Remove Others rows and ? rows and turn financial stress into a number

In [4]:
for column in ['Sleep Duration', 'Dietary Habits', 'Degree']:
    student_data = student_data[student_data[column] != 'Others']
student_data = student_data[student_data['Financial Stress'] != '?'].copy()
student_data['Financial Stress'] = student_data['Financial Stress'].astype(float)

Remove suspicious rows 

In [5]:
student_data = student_data[(student_data['Academic Pressure'] != 0) &
                            (student_data['Study Satisfaction'] != 0) &
                            (student_data['CGPA'] >= 5)]

Dropping unneeded columns, no profession work pressure or job satisfaction since we are focusing on students only, removed id because 
we dont need it for models and suicidal thoughts because that would leak the answer.

In [6]:
student_data = student_data.drop(columns=['Profession', 'Work Pressure', 'Job Satisfaction', 'id', 'Have you ever had suicidal thoughts ?'])
print(student_data.shape)

(27762, 13)


Feature engineering, turning degree column into two new columns, degree level (bachelors, masters, phd) and degree subjects. 
Degree Level 0 and Degree Field Class 12 are the same students. 
    
The dataset does not say whether Degree means the degree a student is studying for now or the highest one they have finished. 
If it means the highest finished, these are students who completed Class 12 and are likely in their first years of college.
If not then we should drop class 12 if we are focusing on college students. 

In [7]:
degree_levels = {
    "'Class 12'": 0,
    'B.Arch': 1, 'B.Com': 1, 'B.Ed': 1, 'B.Pharm': 1, 'B.Tech': 1, 'BA': 1, 'BBA': 1,
    'BCA': 1, 'BE': 1, 'BHM': 1, 'BSc': 1, 'LLB': 1, 'MBBS': 1,
    'LLM': 2, 'M.Com': 2, 'M.Ed': 2, 'M.Pharm': 2, 'M.Tech': 2, 'MA': 2, 'MBA': 2,
    'MCA': 2, 'MD': 2, 'ME': 2, 'MHM': 2, 'MSc': 2,
    'PhD': 3
}

degree_fields = {
    "'Class 12'": 'Class 12',
    'B.Tech': 'Engineering and Computing', 'BE': 'Engineering and Computing',
    'M.Tech': 'Engineering and Computing', 'ME': 'Engineering and Computing',
    'BCA': 'Engineering and Computing', 'MCA': 'Engineering and Computing',
    'MBBS': 'Medicine and Pharmacy', 'MD': 'Medicine and Pharmacy',
    'B.Pharm': 'Medicine and Pharmacy', 'M.Pharm': 'Medicine and Pharmacy',
    'B.Com': 'Business and Commerce', 'M.Com': 'Business and Commerce',
    'BBA': 'Business and Commerce', 'MBA': 'Business and Commerce',
    'B.Ed': 'Education', 'M.Ed': 'Education',
    'LLB': 'Law', 'LLM': 'Law',
    'BSc': 'Sciences', 'MSc': 'Sciences',
    'BA': 'Arts', 'MA': 'Arts',
    'B.Arch': 'Architecture',
    'BHM': 'Hospitality', 'MHM': 'Hospitality',
    'PhD': 'Doctorate'
}

student_data['Degree Level'] = student_data['Degree'].map(degree_levels)
student_data['Degree Field'] = student_data['Degree'].map(degree_fields)
student_data = student_data.drop(columns='Degree')

Our models only accept numbers, so we converted the text columns. Dietary Habits is coded 1 to 3, from unhealthy to healthy, because its depression rate falls steadily (70.8%, 56.0%, 45.3%). Sleep Duration is one hot encoded because its depression rate is uneven at the different levels (59.4% at 7 to 8 hours versus 56.9% at 5 to 6 hours). Coding it 1 to 4 would tell the models the levels are ranked, but depression doesn't follow that ranking, so it wouldn't help the models. Family History is renamed and coded 1 for Yes and 0 for No.

In [8]:
diet_order = {'Unhealthy': 1, 'Moderate': 2, 'Healthy': 3}
yes_no = {'Yes': 1, 'No': 0}

student_data = student_data.rename(columns={'Family History of Mental Illness': 'Family History'})

student_data['Sleep Duration'] = student_data['Sleep Duration'].str.strip("'")
student_data = pd.get_dummies(student_data, columns=['Sleep Duration'], dtype=int)
student_data['Dietary Habits'] = student_data['Dietary Habits'].map(diet_order)
student_data['Family History'] = student_data['Family History'].map(yes_no)

Added age groups (18-21, 22-25, 26-29, 30+) and mapped each city to its region (Northern, Central, Eastern, Western, Southern), so we can compare model results by age and part of the country.

In [9]:
student_data['Age Group'] = pd.cut(student_data['Age'], bins=[17, 21, 25, 29, 60],
                                   labels=['18-21', '22-25', '26-29', '30+'])

region_by_city = {
    'Delhi': 'Northern', 'Faridabad': 'Northern', 'Jaipur': 'Northern',
    'Ludhiana': 'Northern', 'Srinagar': 'Northern',
    'Agra': 'Central', 'Ghaziabad': 'Central', 'Kanpur': 'Central', 'Lucknow': 'Central',
    'Meerut': 'Central', 'Varanasi': 'Central', 'Bhopal': 'Central', 'Indore': 'Central',
    'Kolkata': 'Eastern', 'Patna': 'Eastern',
    'Ahmedabad': 'Western', 'Rajkot': 'Western', 'Surat': 'Western', 'Vadodara': 'Western',
    'Kalyan': 'Western', 'Mumbai': 'Western', 'Nagpur': 'Western', 'Nashik': 'Western',
    'Pune': 'Western', 'Thane': 'Western', 'Vasai-Virar': 'Western',
    'Bangalore': 'Southern', 'Chennai': 'Southern', 'Hyderabad': 'Southern',
    'Visakhapatnam': 'Southern'
}
student_data['Region'] = student_data['City'].map(region_by_city)

The National Family Health Survey (NFHS-5, 2019-21) lists regions instead of cities, so we set each region to its city and merged on city. We added seven columns: women's literacy, women with 10+ years of schooling, people with electricity at home, health insurance coverage, clean cooking fuel, improved sanitation, and sex ratio.

In [10]:
district_data = pd.read_csv('India.csv', low_memory=False)
district_data['State'] = district_data['State'].str.strip()
district_data['District'] = district_data['District'].str.strip()

indicator_names = {
    'Women who are literate (%)': 'Women Literate',
    'Women with 10 or more years of schooling (%)': 'Women 10+ Years Schooling',
    'Population living in households with electricity (%)': 'Households With Electricity',
    'Households with any usual member covered under a health insurance/financing scheme (%)': 'Health Insurance',
    'Households using clean fuel for cooking (%)': 'Clean Cooking Fuel',
    'Population living in households that use an improved sanitation facility (%)': 'Improved Sanitation',
    'Sex ratio of the total population (females per 1,000 males)': 'Sex Ratio'
}
district_data = district_data[district_data['Indicator'].isin(indicator_names.keys())].copy()
district_data['Indicator'] = district_data['Indicator'].map(indicator_names)

district_table = district_data.pivot_table(index=['State', 'District'], columns='Indicator',
                                           values='NFHS 5').reset_index()

city_districts = {
    'Agra': ('Uttar Pradesh', ['Agra']),
    'Ahmedabad': ('Gujarat', ['Ahmedabad']),
    'Bangalore': ('Karnataka', ['Bangalore']),
    'Bhopal': ('Madhya Pradesh', ['Bhopal']),
    'Chennai': ('Tamil Nadu', ['Chennai']),
    'Delhi': ('NCT of Delhi', ['Central', 'East', 'New Delhi', 'North', 'North East', 'North West',
                               'Shahdara', 'South', 'South East', 'South West', 'West']),
    'Faridabad': ('Haryana', ['Faridabad']),
    'Ghaziabad': ('Uttar Pradesh', ['Ghaziabad']),
    'Hyderabad': ('Telangana', ['Hyderabad']),
    'Indore': ('Madhya Pradesh', ['Indore']),
    'Jaipur': ('Rajasthan', ['Jaipur']),
    'Kalyan': ('Maharashtra', ['Thane']),
    'Kanpur': ('Uttar Pradesh', ['Kanpur Nagar']),
    'Kolkata': ('West Bengal', ['Kolkata']),
    'Lucknow': ('Uttar Pradesh', ['Lucknow']),
    'Ludhiana': ('Punjab', ['Ludhiana']),
    'Meerut': ('Uttar Pradesh', ['Meerut']),
    'Mumbai': ('Maharashtra', ['Mumbai', 'Mumbai Suburban']),
    'Nagpur': ('Maharashtra', ['Nagpur']),
    'Nashik': ('Maharashtra', ['Nashik']),
    'Patna': ('Bihar', ['Patna']),
    'Pune': ('Maharashtra', ['Pune']),
    'Rajkot': ('Gujarat', ['Rajkot']),
    'Srinagar': ('Jammu & Kashmir', ['Srinagar']),
    'Surat': ('Gujarat', ['Surat']),
    'Thane': ('Maharashtra', ['Thane']),
    'Vadodara': ('Gujarat', ['Vadodara']),
    'Varanasi': ('Uttar Pradesh', ['Varanasi']),
    'Vasai-Virar': ('Maharashtra', ['Palghar']),
    'Visakhapatnam': ('Andhra Pradesh', ['Visakhapatnam'])
}

city_rows = []
for city, (state, districts) in city_districts.items():
    matching_districts = district_table[(district_table['State'] == state) &
                                        (district_table['District'].isin(districts))]
    city_values = matching_districts[list(indicator_names.values())].mean()
    city_values['City'] = city
    city_rows.append(city_values)
city_table = pd.DataFrame(city_rows)

student_data = student_data.merge(city_table, on='City', how='left')

export cleaned, and splits for different models.

In [11]:
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

os.makedirs('model_ready_data/logistic_regression_neural_network_scaled', exist_ok=True)
os.makedirs('model_ready_data/random_forest_unscaled', exist_ok=True)

model_data = pd.get_dummies(student_data, columns=['Degree Field', 'Gender', 'City', 'Age Group', 'Region'], dtype=int)
model_data.to_csv('model_ready_data/full_cleaned_dataset.csv', index=False)

columns_to_scale = [column for column in model_data.columns
                    if column != 'Depression' and model_data[column].nunique() > 2]

training_data, test_data = train_test_split(model_data, test_size=0.2,
                                            stratify=model_data['Depression'], random_state=42)

scaler = StandardScaler()
training_scaled = training_data.copy()
test_scaled = test_data.copy()
training_scaled[columns_to_scale] = scaler.fit_transform(training_data[columns_to_scale])
test_scaled[columns_to_scale] = scaler.transform(test_data[columns_to_scale])

training_scaled.to_csv('model_ready_data/logistic_regression_neural_network_scaled/train.csv', index=False)
test_scaled.to_csv('model_ready_data/logistic_regression_neural_network_scaled/test.csv', index=False)
training_data.to_csv('model_ready_data/random_forest_unscaled/train.csv', index=False)
test_data.to_csv('model_ready_data/random_forest_unscaled/test.csv', index=False)